**Install Packages**

In [ ]:
!pip -q install kagglehub timm captum lime shap scikit-learn scikit-image opencv-python-headless

**Imports**

In [ ]:
import json, warnings
from pathlib import Path

import cv2
import timm
import shap
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, balanced_accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier

from torch.utils.data import Dataset, DataLoader

from captum.attr import IntegratedGradients, Saliency, NoiseTunnel
from lime import lime_image
from skimage.segmentation import mark_boundaries

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

**Project Folder Names**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_FOLDER = "BRSET XAI Project"

PROCESSED_SHARED_FOLDER = "processed shared"
PREPROCESSED_DATASET_FOLDER = "preprocessed dataset"
MODEL_OUTPUTS_FOLDER = "model outputs"

MODEL_FOLDER = "Multimodal Fusion"

SELECTED_TABULAR_FILENAME = "brset selected tabular.csv"
PREPROCESSING_METADATA_FILENAME = "preprocessing metadata.json"

BEST_MODEL_FILENAME = "best.pt"
LAST_MODEL_FILENAME = "last.pt"
HISTORY_FILENAME = "history.csv"
TEST_PROBS_FILENAME = "test probabilities.npy"
METRICS_FILENAME = "metrics.csv"
CLASSIFICATION_REPORT_FILENAME = "classification report.csv"
CONFUSION_MATRIX_CSV_FILENAME = "confusion matrix.csv"
CONFUSION_MATRIX_FIGURE_FILENAME = "confusion matrix.png"
TRAINING_CURVES_FIGURE_FILENAME = "training curves.png"
GRADCAM_FILENAME = "gradcam.png"
INTEGRATED_GRADIENTS_FILENAME = "integrated gradients.png"
SMOOTHGRAD_FILENAME = "smoothgrad.png"
LIME_FILENAME = "lime.png"
SHAP_CLINICAL_SUMMARY_FILENAME = "shap clinical summary.png"
ABLATION_FILENAME = "modality ablation.csv"
ABLATION_FIGURE_FILENAME = "modality ablation.png"

DATASET_SLUG = "tanzinabdul/fundus-patientwise-split"

**Derived Paths**

In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_FOLDER

PROCESSED_SHARED_DIR = PROJECT_ROOT / PROCESSED_SHARED_FOLDER
PREPROCESSED_DATASET_DIR = PROCESSED_SHARED_DIR / PREPROCESSED_DATASET_FOLDER

MODEL_OUTPUTS_DIR = PROJECT_ROOT / MODEL_OUTPUTS_FOLDER
CURRENT_MODEL_OUTPUT_DIR = MODEL_OUTPUTS_DIR / MODEL_FOLDER

SELECTED_TABULAR_PATH = PREPROCESSED_DATASET_DIR / SELECTED_TABULAR_FILENAME
PREPROCESSING_METADATA_PATH = PREPROCESSED_DATASET_DIR / PREPROCESSING_METADATA_FILENAME

BEST_MODEL_PATH = CURRENT_MODEL_OUTPUT_DIR / BEST_MODEL_FILENAME
LAST_MODEL_PATH = CURRENT_MODEL_OUTPUT_DIR / LAST_MODEL_FILENAME
HISTORY_PATH = CURRENT_MODEL_OUTPUT_DIR / HISTORY_FILENAME
TEST_PROBS_PATH = CURRENT_MODEL_OUTPUT_DIR / TEST_PROBS_FILENAME
METRICS_PATH = CURRENT_MODEL_OUTPUT_DIR / METRICS_FILENAME
CLASSIFICATION_REPORT_PATH = CURRENT_MODEL_OUTPUT_DIR / CLASSIFICATION_REPORT_FILENAME
CONFUSION_MATRIX_CSV_PATH = CURRENT_MODEL_OUTPUT_DIR / CONFUSION_MATRIX_CSV_FILENAME
CONFUSION_MATRIX_FIGURE_PATH = CURRENT_MODEL_OUTPUT_DIR / CONFUSION_MATRIX_FIGURE_FILENAME
TRAINING_CURVES_FIGURE_PATH = CURRENT_MODEL_OUTPUT_DIR / TRAINING_CURVES_FIGURE_FILENAME
GRADCAM_PATH = CURRENT_MODEL_OUTPUT_DIR / GRADCAM_FILENAME
INTEGRATED_GRADIENTS_PATH = CURRENT_MODEL_OUTPUT_DIR / INTEGRATED_GRADIENTS_FILENAME
SMOOTHGRAD_PATH = CURRENT_MODEL_OUTPUT_DIR / SMOOTHGRAD_FILENAME
LIME_PATH = CURRENT_MODEL_OUTPUT_DIR / LIME_FILENAME
SHAP_CLINICAL_SUMMARY_PATH = CURRENT_MODEL_OUTPUT_DIR / SHAP_CLINICAL_SUMMARY_FILENAME
ABLATION_PATH = CURRENT_MODEL_OUTPUT_DIR / ABLATION_FILENAME
ABLATION_FIGURE_PATH = CURRENT_MODEL_OUTPUT_DIR / ABLATION_FIGURE_FILENAME

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PREPROCESSED_DATASET_DIR:", PREPROCESSED_DATASET_DIR)
print("CURRENT_MODEL_OUTPUT_DIR:", CURRENT_MODEL_OUTPUT_DIR)

**Kaggle Download**

In [ ]:
import kagglehub
from pathlib import Path

dataset_path = kagglehub.dataset_download(DATASET_SLUG)

RAW_DIR = Path(dataset_path)

print("Dataset downloaded successfully.")
print("RAW_DIR:", RAW_DIR)

**Model Config**

In [ ]:
import sys
SRC_DIR = PROJECT_ROOT / "src"
sys.path.append(str(SRC_DIR))

In [ ]:
from funduspreprocessing import preprocess_fundus
from model_common import (
    SEED,
    IMG_SIZE,
    BATCH_SIZE,
    EPOCHS,
    LR,
    WEIGHT_DECAY,
    NUM_WORKERS,
    ROTATION_DEGREES,
    set_seed,
    build_image_transforms,
    denormalize_image_tensor,
    predict_model,
    evaluate_predictions,
    save_metrics,
    save_probabilities,
    save_classification_report,
    save_confusion_matrix_csv,
    train_with_validation,
)

set_seed(SEED)

preprocessing_metadata = json.load(open(PREPROCESSING_METADATA_PATH))
TARGET_COL = preprocessing_metadata["target"]["target_column"]
selected_feature_columns = preprocessing_metadata["feature_columns"]["selected_features"]
cohort_columns = preprocessing_metadata["feature_columns"]["cohort_columns"]

NUM_WORKERS = 0

**Load Processed Data**

In [ ]:
df = pd.read_csv(SELECTED_TABULAR_PATH)

IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp"]
image_files = [
    path
    for path in RAW_DIR.rglob("*")
    if path.suffix.lower() in IMAGE_EXTENSIONS
]

image_path_map = {
    path.stem: str(path)
    for path in image_files
}

df["processed_image_path"] = df["processed_image_path"].apply(
    lambda x: image_path_map[Path(str(x)).stem]
)

df[TARGET_COL] = df[TARGET_COL].astype(int)

display(df.head())
print(df.shape)
print(df["split"].value_counts())
print(df[TARGET_COL].value_counts().sort_index())
print("Selected features:", selected_feature_columns)
print("Cohort columns:", cohort_columns)

**Dataset Class**

In [ ]:
train_tfms, test_tfms = build_image_transforms(
    rotation_degrees=ROTATION_DEGREES
)

class BRSETMultimodalDataset(Dataset):
    def __init__(self, data, feature_columns, transform=None):
        self.data = data.reset_index(drop=True)
        self.feature_columns = feature_columns
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        img = preprocess_fundus(
            row["processed_image_path"],
            img_size=IMG_SIZE
        )

        img = Image.fromarray(img)

        if self.transform:
            img = self.transform(img)

        tab = torch.tensor(
            row[self.feature_columns].values.astype(np.float32),
            dtype=torch.float32
        )

        y = torch.tensor(int(row[TARGET_COL]), dtype=torch.long)

        return img, tab, y

**DataLoaders**

In [ ]:
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

num_classes = int(df[TARGET_COL].max() + 1)

tabular_input_dim = len(selected_feature_columns)

train_ds = BRSETMultimodalDataset(train_df, selected_feature_columns, transform=train_tfms)
val_ds = BRSETMultimodalDataset(val_df, selected_feature_columns, transform=test_tfms)
test_ds = BRSETMultimodalDataset(test_df, selected_feature_columns, transform=test_tfms)

pin_memory = DEVICE.type == "cuda"

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=train_df[TARGET_COL].to_numpy()
)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print("Classes:", num_classes)
print("Tabular input dimension:", tabular_input_dim)
print("Class weights:", class_weights)

**Multimodal Batch Loading Timing Test**

In [ ]:
import time

start = time.time()

debug_batch = next(iter(train_loader))

debug_imgs, debug_tabs, debug_labels = debug_batch

elapsed = time.time() - start

print("One multimodal batch loaded successfully.")
print("Image batch shape:", debug_imgs.shape)
print("Tabular batch shape:", debug_tabs.shape)
print("Label batch shape:", debug_labels.shape)
print("Time to load one batch:", round(elapsed, 2), "seconds")

**Multimodal Fusion Model**

In [ ]:
class ImageTabularFusionModel(nn.Module):
    def __init__(self, backbone_name="resnet50", tabular_dim=10, num_classes=5):
        super().__init__()

        self.image_backbone = timm.create_model(
            backbone_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.tabular_branch = nn.Sequential(
            nn.Linear(tabular_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.30)
        )

        self.classifier = nn.Sequential(
            nn.Linear(image_dim + 128, 512),
            nn.ReLU(),
            nn.Dropout(0.40),
            nn.Linear(512, num_classes)
        )

    def forward(self, image, tabular):
        image_features = self.image_backbone(image)
        tabular_features = self.tabular_branch(tabular)
        fused_features = torch.cat([image_features, tabular_features], dim=1)
        return self.classifier(fused_features)

fusion_model = ImageTabularFusionModel(
    backbone_name="resnet50",
    tabular_dim=tabular_input_dim,
    num_classes=num_classes
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    fusion_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

**Multimodal Forward Timing Test**

In [ ]:
start = time.time()

fusion_model.eval()

debug_imgs = debug_imgs.to(DEVICE)
debug_tabs = debug_tabs.to(DEVICE)

with torch.no_grad():
    debug_logits = fusion_model(debug_imgs, debug_tabs)

elapsed = time.time() - start

print("One multimodal forward pass completed.")
print("Logits shape:", debug_logits.shape)
print("Time for one forward pass:", round(elapsed, 2), "seconds")

**Train Model**

In [ ]:
RESUME_TRAINING = False

history, best_macro_f1 = train_with_validation(
    model=fusion_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=DEVICE,
    epochs=EPOCHS,
    best_model_path=BEST_MODEL_PATH,
    last_model_path=LAST_MODEL_PATH,
    history_path=HISTORY_PATH,
    resume_training=RESUME_TRAINING,
    monitor_metric="macro_f1"
)

**Training Curves**

In [ ]:
history_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df["epoch"], history_df["loss"], marker="o")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(history_df["epoch"], history_df["val_macro_f1"], marker="o", label="Macro F1")
axes[1].plot(history_df["epoch"], history_df["val_bal_acc"], marker="o", label="Balanced Accuracy")
axes[1].set_title("Validation Metrics")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].legend()

plt.tight_layout()
plt.savefig(TRAINING_CURVES_FIGURE_PATH, dpi=300, bbox_inches="tight")
plt.show()

**Test Fusion Model**

In [ ]:
fusion_model.load_state_dict(
    torch.load(BEST_MODEL_PATH, map_location=DEVICE)
)

y_true, y_pred, y_prob, infer_time = predict_model(
    fusion_model,
    test_loader,
    DEVICE
)

fusion_metrics = evaluate_predictions(
    y_true=y_true,
    y_pred=y_pred,
    y_prob=y_prob,
    model_name="Multimodal Fusion",
    inference_time_seconds=infer_time
)

save_probabilities(y_prob, TEST_PROBS_PATH)
save_metrics(fusion_metrics, METRICS_PATH)

save_classification_report(
    y_true,
    y_pred,
    CLASSIFICATION_REPORT_PATH
)

save_confusion_matrix_csv(
    y_true,
    y_pred,
    CONFUSION_MATRIX_CSV_PATH
)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Multimodal Fusion Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_FIGURE_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(fusion_metrics)

**XAI Helper: Denormalization**

In [ ]:
xai_sample_index = int(np.argmax(y_prob.max(axis=1)))
sample_img, sample_tab, sample_y = test_ds[xai_sample_index]

img_np = denormalize_image_tensor(sample_img)

plt.imshow(img_np)
plt.title(f"Label: {sample_y}")
plt.axis("off")
plt.show()

**XAI 1: Grad-CAM**

In [ ]:
def get_last_conv_layer(model):
    for name, module in reversed(list(model.named_modules())):
        if isinstance(module, nn.Conv2d):
            return name, module
    return None, None


def gradcam(model, img_tensor, tab_tensor, class_idx=None):
    model.eval()

    layer_name, target_layer = get_last_conv_layer(model)
    activations = []
    gradients = []

    def forward_hook(module, inp, out):
        activations.append(out)

    def backward_hook(module, grad_in, grad_out):
        gradients.append(grad_out[0])

    h1 = target_layer.register_forward_hook(forward_hook)
    h2 = target_layer.register_full_backward_hook(backward_hook)

    img = img_tensor.unsqueeze(0).to(DEVICE)
    tab = tab_tensor.unsqueeze(0).to(DEVICE)

    output = model(img, tab)

    if class_idx is None:
        class_idx = output.argmax(1).item()

    score = output[:, class_idx]
    model.zero_grad()
    score.backward()

    grads = gradients[0]
    acts = activations[0]

    weights = grads.mean(dim=(2, 3), keepdim=True)
    cam = (weights * acts).sum(dim=1).squeeze()
    cam = F.relu(cam)
    cam = cam.detach().cpu().numpy()
    cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    h1.remove()
    h2.remove()

    return cam

cam = gradcam(fusion_model, sample_img, sample_tab)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_np)
plt.axis("off")
plt.title("Original")

plt.subplot(1, 2, 2)
plt.imshow(img_np)
plt.imshow(cam, alpha=0.45)
plt.axis("off")
plt.title("Grad-CAM")
plt.tight_layout()
plt.savefig(GRADCAM_PATH, dpi=300, bbox_inches="tight")
plt.show()

**XAI 2: Integrated Gradients**

In [ ]:
def integrated_gradients_visual(model, img_tensor, tab_tensor):
    model.eval()

    img_input = img_tensor.unsqueeze(0).to(DEVICE)
    tab_input = tab_tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred_class = model(img_input, tab_input).argmax(1).item()

    def forward_func(x):
        return model(x, tab_input)

    ig = IntegratedGradients(forward_func)

    attr = ig.attribute(
        img_input,
        target=pred_class,
        n_steps=32
    )

    attr = attr.squeeze().detach().cpu().numpy()
    attr = np.abs(attr).mean(axis=0)
    attr = (attr - attr.min()) / (attr.max() - attr.min() + 1e-8)

    return attr

ig_map = integrated_gradients_visual(fusion_model, sample_img, sample_tab)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_np)
plt.axis("off")
plt.title("Original")

plt.subplot(1, 2, 2)
plt.imshow(img_np)
plt.imshow(ig_map, alpha=0.45)
plt.axis("off")
plt.title("Integrated Gradients")
plt.tight_layout()
plt.savefig(INTEGRATED_GRADIENTS_PATH, dpi=300, bbox_inches="tight")
plt.show()

**XAI 3: SmoothGrad**

In [ ]:
def smoothgrad_visual(model, img_tensor, tab_tensor):
    model.eval()

    img_input = img_tensor.unsqueeze(0).to(DEVICE)
    tab_input = tab_tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred_class = model(img_input, tab_input).argmax(1).item()

    def forward_func(x):
        return model(x, tab_input)

    saliency = Saliency(forward_func)
    nt = NoiseTunnel(saliency)

    attr = nt.attribute(
        img_input,
        target=pred_class,
        nt_type="smoothgrad",
        nt_samples=20,
        stdevs=0.15
    )

    attr = attr.squeeze().detach().cpu().numpy()
    attr = np.abs(attr).mean(axis=0)
    attr = (attr - attr.min()) / (attr.max() - attr.min() + 1e-8)

    return attr

sg_map = smoothgrad_visual(fusion_model, sample_img, sample_tab)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_np)
plt.axis("off")
plt.title("Original")

plt.subplot(1, 2, 2)
plt.imshow(img_np)
plt.imshow(sg_map, alpha=0.45)
plt.axis("off")
plt.title("SmoothGrad")
plt.tight_layout()
plt.savefig(SMOOTHGRAD_PATH, dpi=300, bbox_inches="tight")
plt.show()

**XAI 4: LIME Image**

In [ ]:
def lime_predict(images_np):
    fusion_model.eval()

    batch_imgs = []

    for im in images_np:
        im = Image.fromarray((im * 255).astype(np.uint8))
        batch_imgs.append(test_tfms(im))

    batch_imgs = torch.stack(batch_imgs).to(DEVICE)
    batch_tabs = sample_tab.unsqueeze(0).repeat(batch_imgs.size(0), 1).to(DEVICE)

    with torch.no_grad():
        probs = torch.softmax(
            fusion_model(batch_imgs, batch_tabs),
            dim=1
        ).cpu().numpy()

    return probs

explainer = lime_image.LimeImageExplainer()

explanation = explainer.explain_instance(
    img_np,
    lime_predict,
    top_labels=1,
    hide_color=0,
    num_samples=500
)

temp, mask = explanation.get_image_and_mask(
    explanation.top_labels[0],
    positive_only=True,
    num_features=8,
    hide_rest=False
)

plt.figure(figsize=(6, 6))
plt.imshow(mark_boundaries(temp, mask))
plt.title("LIME Explanation")
plt.axis("off")
plt.tight_layout()
plt.savefig(LIME_PATH, dpi=300, bbox_inches="tight")
plt.show()

**XAI 5: SHAP Clinical Feature Surrogate Explanation**

In [ ]:
sample_for_shap = test_df.sample(
    min(500, len(test_df)),
    random_state=SEED
)

X_train_shap = train_df[selected_feature_columns].astype(np.float32)
X_shap = sample_for_shap[selected_feature_columns].astype(np.float32)

rf_explainer_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1
)

rf_explainer_model.fit(
    X_train_shap,
    train_df[TARGET_COL]
)

shap_explainer = shap.TreeExplainer(rf_explainer_model)
shap_values = shap_explainer.shap_values(X_shap)

if isinstance(shap_values, list):
    shap_importance_values = np.mean(
        [
            np.abs(class_shap_values).mean(axis=0)
            for class_shap_values in shap_values
        ],
        axis=0
    )
else:
    shap_values_array = np.array(shap_values)

    if shap_values_array.ndim == 3:
        shap_importance_values = np.abs(shap_values_array).mean(axis=(0, 2))
    else:
        shap_importance_values = np.abs(shap_values_array).mean(axis=0)

shap_importance_df = pd.DataFrame({
    "feature": selected_feature_columns,
    "shap_importance": shap_importance_values
}).sort_values("shap_importance", ascending=False)

plt.figure(figsize=(8, 5))
plt.barh(
    shap_importance_df["feature"][::-1],
    shap_importance_df["shap_importance"][::-1]
)
plt.title("SHAP Clinical Feature Importance Surrogate")
plt.xlabel("Mean Absolute SHAP Value")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(SHAP_CLINICAL_SUMMARY_PATH, dpi=300, bbox_inches="tight")
plt.show()

display(shap_importance_df)

**Modality Ablation for Fusion Model**

In [ ]:
@torch.no_grad()
def fusion_ablation_test(model, loader):
    model.eval()

    true_all = []
    pred_full = []
    pred_zero_tabular = []
    pred_zero_image = []

    for imgs, tabs, y in tqdm(loader):
        imgs = imgs.to(DEVICE)
        tabs = tabs.to(DEVICE)

        full_logits = model(imgs, tabs)
        zero_tabular_logits = model(imgs, torch.zeros_like(tabs))
        zero_image_logits = model(torch.zeros_like(imgs), tabs)

        true_all.append(y.numpy())
        pred_full.append(full_logits.argmax(1).cpu().numpy())
        pred_zero_tabular.append(zero_tabular_logits.argmax(1).cpu().numpy())
        pred_zero_image.append(zero_image_logits.argmax(1).cpu().numpy())

    true_all = np.concatenate(true_all)
    pred_full = np.concatenate(pred_full)
    pred_zero_tabular = np.concatenate(pred_zero_tabular)
    pred_zero_image = np.concatenate(pred_zero_image)

    return pd.DataFrame({
        "setting": [
            "full_model",
            "image_only_zero_tabular",
            "tabular_only_zero_image"
        ],
        "balanced_accuracy": [
            balanced_accuracy_score(true_all, pred_full),
            balanced_accuracy_score(true_all, pred_zero_tabular),
            balanced_accuracy_score(true_all, pred_zero_image)
        ],
        "macro_f1": [
            f1_score(true_all, pred_full, average="macro", zero_division=0),
            f1_score(true_all, pred_zero_tabular, average="macro", zero_division=0),
            f1_score(true_all, pred_zero_image, average="macro", zero_division=0)
        ]
    })

ablation_df = fusion_ablation_test(fusion_model, test_loader)

ablation_df.to_csv(ABLATION_PATH, index=False)

display(ablation_df)

x = np.arange(len(ablation_df))
width = 0.35

plt.figure(figsize=(9, 4))
plt.bar(x - width / 2, ablation_df["balanced_accuracy"], width, label="Balanced Accuracy")
plt.bar(x + width / 2, ablation_df["macro_f1"], width, label="Macro F1")
plt.xticks(x, ablation_df["setting"], rotation=20, ha="right")
plt.ylabel("Score")
plt.title("Modality Ablation")
plt.legend()
plt.tight_layout()
plt.savefig(ABLATION_FIGURE_PATH, dpi=300, bbox_inches="tight")
plt.show()